# NB_bishop_ch05_figures

**Bishop Chapter 5 — Key Figures: Discriminant Functions, Logistic Regression, Softmax, Fisher's Linear Discriminant & Decision Boundaries**

This notebook generates pedagogical matplotlib figures for Bishop Chapter 5 on single-layer networks for classification.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_05/NB_bishop_ch05_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.colors import ListedColormap
import os

# ── colour palette ──────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# ── global rcParams ─────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.loc']         = 'upper center'
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Setup complete.')

## Figure 5.1 — Linear Decision Boundaries (Two-Class)

In [ ]:
np.random.seed(42)

# Generate two-class data
N = 60
X_c1 = np.random.randn(N, 2) * 0.8 + np.array([-1.5, -1.0])
X_c2 = np.random.randn(N, 2) * 0.8 + np.array([1.5, 1.0])

# Decision boundary: w^T x + w0 = 0
# Simple linear separator
w = np.array([1.0, 0.8])
w0 = 0.0

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X_c1[:, 0], X_c1[:, 1], c=COLORS['blue'], s=50, edgecolor='k',
           linewidth=0.5, label='$\\mathcal{C}_1$', zorder=3)
ax.scatter(X_c2[:, 0], X_c2[:, 1], c=COLORS['red'], s=50, edgecolor='k',
           linewidth=0.5, label='$\\mathcal{C}_2$', zorder=3)

# Decision boundary line
x_line = np.linspace(-4, 4, 200)
y_line = -(w[0] * x_line + w0) / w[1]
ax.plot(x_line, y_line, color=COLORS['green'], lw=2.5, label='Decision boundary')

# Shade regions
xx, yy = np.meshgrid(np.linspace(-4, 4, 300), np.linspace(-4, 4, 300))
Z = w[0] * xx + w[1] * yy + w0
ax.contourf(xx, yy, Z, levels=[-100, 0, 100],
            colors=[COLORS['blue'], COLORS['red']], alpha=0.06)

# Annotate weight vector (normal to boundary)
mid_x, mid_y = 0.0, 0.0
w_norm = w / np.linalg.norm(w)
ax.annotate('', xy=(mid_x + 1.2 * w_norm[0], mid_y + 1.2 * w_norm[1]),
            xytext=(mid_x, mid_y),
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2))
ax.text(mid_x + 1.4 * w_norm[0], mid_y + 1.4 * w_norm[1] + 0.2,
        '$\\mathbf{w}$', fontsize=14, color=COLORS['green'], fontweight='bold')

ax.annotate('$\\mathcal{R}_1$', xy=(-2.5, 2.5), fontsize=16,
            color=COLORS['blue'], fontweight='bold')
ax.annotate('$\\mathcal{R}_2$', xy=(2.0, -2.5), fontsize=16,
            color=COLORS['red'], fontweight='bold')

ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Linear Decision Boundary: $\\mathbf{w}^\\top \\mathbf{x} + w_0 = 0$')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig5_1_decision_boundaries')
plt.show()

## Figure 5.2 — Multi-Class Decision Regions

In [ ]:
np.random.seed(7)

# Three-class data in 2D
N = 50
centers = [np.array([-2.0, 0.0]), np.array([2.0, 0.0]), np.array([0.0, 2.5])]
class_colors = [COLORS['blue'], COLORS['red'], COLORS['green']]
class_labels = ['$\\mathcal{C}_1$', '$\\mathcal{C}_2$', '$\\mathcal{C}_3$']

X_classes = []
for c in centers:
    X_classes.append(np.random.randn(N, 2) * 0.7 + c)

# Linear discriminant weights (one-vs-all style for 3 classes)
# w_k^T x + w_{k0} for each class k; assign x to argmax_k
W = np.array([[-1.5, -0.3],   # class 1 favours left
              [ 1.5, -0.3],   # class 2 favours right
              [ 0.0,  1.8]])  # class 3 favours top
b = np.array([0.0, 0.0, -1.0])

fig, ax = plt.subplots(figsize=(8, 7))

# Color-coded decision regions
xx, yy = np.meshgrid(np.linspace(-5, 5, 400), np.linspace(-3, 6, 400))
grid = np.c_[xx.ravel(), yy.ravel()]
scores = grid @ W.T + b
pred = np.argmax(scores, axis=1).reshape(xx.shape)

region_cmap = ListedColormap([COLORS['blue'], COLORS['red'], COLORS['green']])
ax.contourf(xx, yy, pred, levels=[-0.5, 0.5, 1.5, 2.5],
            colors=[COLORS['blue'], COLORS['red'], COLORS['green']], alpha=0.10)

# Decision boundaries (pairwise)
for i in range(3):
    for j in range(i + 1, 3):
        dw = W[i] - W[j]
        db = b[i] - b[j]
        x_bd = np.linspace(-5, 5, 200)
        # dw[0]*x + dw[1]*y + db = 0 => y = -(dw[0]*x + db) / dw[1]
        if abs(dw[1]) > 1e-8:
            y_bd = -(dw[0] * x_bd + db) / dw[1]
            mask = (y_bd > -3) & (y_bd < 6)
            ax.plot(x_bd[mask], y_bd[mask], 'k-', lw=1.5, alpha=0.6)

# Scatter data
for k in range(3):
    ax.scatter(X_classes[k][:, 0], X_classes[k][:, 1],
               c=class_colors[k], s=45, edgecolor='k', linewidth=0.5,
               label=class_labels[k], zorder=3)

# Region labels
ax.text(-3.5, 1.5, '$\\mathcal{R}_1$', fontsize=18, color=COLORS['blue'], fontweight='bold')
ax.text(3.0, 1.5, '$\\mathcal{R}_2$', fontsize=18, color=COLORS['red'], fontweight='bold')
ax.text(0.0, 5.0, '$\\mathcal{R}_3$', fontsize=18, color=COLORS['green'], fontweight='bold')

ax.set_xlim(-5, 5)
ax.set_ylim(-3, 6)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Multi-Class Decision Regions ($K=3$, Linear Discriminants)')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig5_2_decision_regions')
plt.show()

## Figure 5.4 — Logistic Sigmoid Function

In [ ]:
a = np.linspace(-8, 8, 500)
sigma = 1 / (1 + np.exp(-a))

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(a, sigma, color=COLORS['blue'], lw=2.5, label='$\\sigma(a) = \\frac{1}{1+e^{-a}}$')

# Reference lines
ax.axhline(0.5, ls=':', color='gray', lw=1, alpha=0.6)
ax.axhline(1.0, ls=':', color='gray', lw=1, alpha=0.4)
ax.axhline(0.0, ls=':', color='gray', lw=1, alpha=0.4)
ax.axvline(0.0, ls=':', color='gray', lw=1, alpha=0.6)

# Midpoint annotation
ax.plot(0, 0.5, 'o', ms=8, color=COLORS['red'], zorder=5, markeredgecolor='k', markeredgewidth=0.8)
ax.annotate('Midpoint $\\sigma(0)=0.5$', xy=(0, 0.5), xytext=(2.5, 0.35),
            fontsize=11, color=COLORS['red'],
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.5))

# Slope annotation at midpoint
ax.annotate('Max slope $= \\frac{1}{4}$', xy=(0.3, 0.57), xytext=(3.5, 0.75),
            fontsize=11, color=COLORS['green'],
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=1.5))

# Draw tangent line at a=0 (slope = 0.25)
a_tan = np.linspace(-2, 2, 100)
y_tan = 0.5 + 0.25 * a_tan
ax.plot(a_tan, y_tan, '--', color=COLORS['green'], lw=1.5, alpha=0.7)

# Saturation annotations
ax.annotate('Saturation: $\\sigma \\to 1$', xy=(6, 0.98), fontsize=10,
            color=COLORS['amber'], ha='center', style='italic')
ax.annotate('Saturation: $\\sigma \\to 0$', xy=(-6, 0.02), fontsize=10,
            color=COLORS['amber'], ha='center', style='italic')

ax.set_xlabel('$a$')
ax.set_ylabel('$\\sigma(a)$')
ax.set_title('Logistic Sigmoid Function')
ax.set_ylim(-0.08, 1.12)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig5_4_logistic_sigmoid')
plt.show()

## Figure 5.5 — Softmax Visualization

In [ ]:
# Softmax: input logits -> output probabilities for K=4 classes
logits = np.array([2.0, 1.0, 0.5, -1.0])
K = len(logits)

def softmax(z):
    e = np.exp(z - np.max(z))  # numerically stable
    return e / e.sum()

probs = softmax(logits)
class_names = ['Class 1', 'Class 2', 'Class 3', 'Class 4']
bar_colors = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['amber']]

fig, axes = plt.subplots(1, 3, figsize=(14, 5),
                         gridspec_kw={'width_ratios': [1, 0.4, 1]})

# Left: logits
ax = axes[0]
bars = ax.barh(class_names, logits, color=bar_colors, edgecolor='k', linewidth=0.8)
for bar, val in zip(bars, logits):
    ax.text(val + 0.1 if val >= 0 else val - 0.1, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', ha='left' if val >= 0 else 'right',
            fontsize=12, fontweight='bold')
ax.set_xlabel('Logit value $a_k$')
ax.set_title('Input Logits')
ax.set_xlim(-2, 3)
ax.invert_yaxis()

# Middle: arrow
ax = axes[1]
ax.axis('off')
ax.annotate('', xy=(0.9, 0.5), xytext=(0.1, 0.5),
            arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=3,
                            mutation_scale=25),
            xycoords='axes fraction')
ax.text(0.5, 0.65, 'softmax', fontsize=13, ha='center', va='center',
        transform=ax.transAxes, fontweight='bold', color=COLORS['blue'])
ax.text(0.5, 0.35, '$\\frac{e^{a_k}}{\\sum_j e^{a_j}}$', fontsize=16,
        ha='center', va='center', transform=ax.transAxes, color=COLORS['blue'])

# Right: probabilities
ax = axes[2]
bars = ax.barh(class_names, probs, color=bar_colors, edgecolor='k', linewidth=0.8)
for bar, val in zip(bars, probs):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha='left', fontsize=12, fontweight='bold')
ax.set_xlabel('Probability $p_k$')
ax.set_title('Output Probabilities')
ax.set_xlim(0, 0.7)
ax.invert_yaxis()

# Verify they sum to 1
ax.text(0.55, 0.95, f'$\\sum p_k = {probs.sum():.3f}$', fontsize=11,
        transform=ax.transAxes, va='top', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.8))

fig.suptitle('Softmax Function ($K=4$ Classes)', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig5_5_softmax')
plt.show()

## Figure 5.7 — Fisher's Linear Discriminant

In [ ]:
np.random.seed(12)

# Generate two-class data with correlated features
N1, N2 = 80, 80
cov1 = np.array([[1.2, 0.8], [0.8, 1.0]])
cov2 = np.array([[1.0, -0.5], [-0.5, 1.2]])
mu1 = np.array([-1.5, -1.0])
mu2 = np.array([2.0, 1.5])

X1 = np.random.multivariate_normal(mu1, cov1, N1)
X2 = np.random.multivariate_normal(mu2, cov2, N2)

# Compute Fisher's direction
m1 = X1.mean(axis=0)
m2 = X2.mean(axis=0)
S1 = (X1 - m1).T @ (X1 - m1)
S2 = (X2 - m2).T @ (X2 - m2)
S_W = S1 + S2  # within-class scatter
w_fisher = np.linalg.solve(S_W, m2 - m1)
w_fisher = w_fisher / np.linalg.norm(w_fisher)

# Project data onto Fisher direction
proj1 = X1 @ w_fisher
proj2 = X2 @ w_fisher

fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                         gridspec_kw={'width_ratios': [1.2, 1]})

# Left panel: 2D scatter with projection direction
ax = axes[0]
ax.scatter(X1[:, 0], X1[:, 1], c=COLORS['blue'], s=30, alpha=0.6,
           edgecolor='k', linewidth=0.3, label='$\\mathcal{C}_1$')
ax.scatter(X2[:, 0], X2[:, 1], c=COLORS['red'], s=30, alpha=0.6,
           edgecolor='k', linewidth=0.3, label='$\\mathcal{C}_2$')

# Means
ax.plot(*m1, 'D', ms=10, color=COLORS['blue'], markeredgecolor='k',
        markeredgewidth=1.5, zorder=5)
ax.plot(*m2, 'D', ms=10, color=COLORS['red'], markeredgecolor='k',
        markeredgewidth=1.5, zorder=5)

# Fisher direction line (extend through origin area)
t = np.linspace(-5, 7, 200)
line_x = t * w_fisher[0]
line_y = t * w_fisher[1]
ax.plot(line_x, line_y, color=COLORS['green'], lw=2.5, ls='--',
        label='Fisher direction $\\mathbf{w}$', zorder=2)

# Arrow showing w direction
origin = np.array([0, 0])
ax.annotate('', xy=origin + 2.5 * w_fisher, xytext=origin,
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2.5))
ax.text(2.5 * w_fisher[0] + 0.3, 2.5 * w_fisher[1] + 0.3,
        '$\\mathbf{w}$', fontsize=14, color=COLORS['green'], fontweight='bold')

# Show projections as ticks on the line
for xi in X1:
    proj_pt = (xi @ w_fisher) * w_fisher
    ax.plot([xi[0], proj_pt[0]], [xi[1], proj_pt[1]],
            color=COLORS['blue'], lw=0.3, alpha=0.25)

for xi in X2:
    proj_pt = (xi @ w_fisher) * w_fisher
    ax.plot([xi[0], proj_pt[0]], [xi[1], proj_pt[1]],
            color=COLORS['red'], lw=0.3, alpha=0.25)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title("Fisher's Linear Discriminant (2D)")
ax.set_xlim(-5, 6)
ax.set_ylim(-5, 6)
ax.set_aspect('equal')
ax.legend()

# Right panel: 1D histograms of projections
ax = axes[1]
bins = np.linspace(-5, 7, 40)
ax.hist(proj1, bins=bins, color=COLORS['blue'], alpha=0.6, edgecolor='k',
        linewidth=0.5, label='$\\mathcal{C}_1$ projected', density=True)
ax.hist(proj2, bins=bins, color=COLORS['red'], alpha=0.6, edgecolor='k',
        linewidth=0.5, label='$\\mathcal{C}_2$ projected', density=True)

# Projected means
ax.axvline(proj1.mean(), color=COLORS['blue'], lw=2, ls='--')
ax.axvline(proj2.mean(), color=COLORS['red'], lw=2, ls='--')

# Threshold
threshold = (proj1.mean() + proj2.mean()) / 2
ax.axvline(threshold, color=COLORS['green'], lw=2.5, ls='-',
           label=f'Threshold = {threshold:.2f}')

ax.set_xlabel('Projection onto $\\mathbf{w}$')
ax.set_ylabel('Density')
ax.set_title('Projected Distributions')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig5_7_fisher_discriminant')
plt.show()

## Figure 5.10 — Cross-Entropy Loss vs MSE Loss

In [ ]:
# Cross-entropy vs MSE loss when true label t=1
p = np.linspace(0.001, 0.999, 500)

# Cross-entropy: -t*log(p) - (1-t)*log(1-p), with t=1 => -log(p)
ce_loss = -np.log(p)

# MSE: (p - t)^2, with t=1 => (1 - p)^2
mse_loss = (1 - p) ** 2

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(p, ce_loss, color=COLORS['blue'], lw=2.5, label='Cross-entropy: $-\\ln p$')
ax.plot(p, mse_loss, color=COLORS['red'], lw=2.5, label='MSE: $(1-p)^2$')

# Annotations
ax.annotate('CE penalises confident\nwrong predictions heavily',
            xy=(0.05, -np.log(0.05)), xytext=(0.25, 3.5),
            fontsize=10, color=COLORS['blue'],
            arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=1.5))

ax.annotate('MSE saturates\n(weak gradient)',
            xy=(0.05, (1 - 0.05)**2), xytext=(0.3, 1.5),
            fontsize=10, color=COLORS['red'],
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.5))

# Mark optimal point
ax.plot(1.0, 0.0, 'o', ms=8, color=COLORS['green'], zorder=5,
        markeredgecolor='k', markeredgewidth=0.8)
ax.annotate('$p=1$ (correct)', xy=(0.95, 0.1), fontsize=10,
            color=COLORS['green'], ha='right')

ax.set_xlabel('Predicted probability $p$ for true class ($t=1$)')
ax.set_ylabel('Loss')
ax.set_title('Cross-Entropy vs MSE Loss')
ax.set_ylim(-0.1, 5.0)
ax.set_xlim(0, 1.05)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig5_10_cross_entropy')
plt.show()

## Figure 5.13 — Logistic Regression on 2D Data

In [ ]:
np.random.seed(99)

# Generate 2D binary classification data
N = 80
X_pos = np.random.randn(N, 2) * 0.9 + np.array([1.5, 1.5])
X_neg = np.random.randn(N, 2) * 0.9 + np.array([-1.5, -1.5])
X_all = np.vstack([X_pos, X_neg])
y_all = np.concatenate([np.ones(N), np.zeros(N)])

# Fit logistic regression via IRLS (iteratively reweighted least squares)
# Add bias column
Phi = np.c_[np.ones(2 * N), X_all]

# Initialize weights
w_lr = np.zeros(3)
for _ in range(20):  # IRLS iterations
    a = Phi @ w_lr
    y_pred = 1 / (1 + np.exp(-a))
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    R = np.diag(y_pred * (1 - y_pred))
    # w_new = w - (Phi^T R Phi)^{-1} Phi^T (y_pred - y)
    H = Phi.T @ R @ Phi + 1e-6 * np.eye(3)
    grad = Phi.T @ (y_pred - y_all)
    w_lr = w_lr - np.linalg.solve(H, grad)

fig, ax = plt.subplots(figsize=(8, 7))

# Probability gradient (background)
xx, yy = np.meshgrid(np.linspace(-5, 5, 300), np.linspace(-5, 5, 300))
grid = np.c_[np.ones(xx.size), xx.ravel(), yy.ravel()]
prob_grid = 1 / (1 + np.exp(-(grid @ w_lr)))
prob_grid = prob_grid.reshape(xx.shape)

# Filled contour for probability
from matplotlib.colors import LinearSegmentedColormap
prob_cmap = LinearSegmentedColormap.from_list('prob',
    [COLORS['blue'], 'white', COLORS['red']], N=256)
cf = ax.contourf(xx, yy, prob_grid, levels=np.linspace(0, 1, 21),
                 cmap=prob_cmap, alpha=0.35)
cbar = fig.colorbar(cf, ax=ax, shrink=0.8, label='$P(\\mathcal{C}_1 | \\mathbf{x})$')

# Decision boundary (p = 0.5)
ax.contour(xx, yy, prob_grid, levels=[0.5], colors=[COLORS['green']],
           linewidths=2.5, linestyles='-')

# Probability contours
ax.contour(xx, yy, prob_grid, levels=[0.1, 0.25, 0.75, 0.9],
           colors=['gray'], linewidths=0.8, linestyles=':', alpha=0.6)

# Data points
ax.scatter(X_neg[:, 0], X_neg[:, 1], c=COLORS['blue'], s=40, edgecolor='k',
           linewidth=0.5, label='$\\mathcal{C}_0$', zorder=3)
ax.scatter(X_pos[:, 0], X_pos[:, 1], c=COLORS['red'], s=40, edgecolor='k',
           linewidth=0.5, label='$\\mathcal{C}_1$', zorder=3)

# Label the decision boundary
ax.text(2.8, -2.2, '$p = 0.5$', fontsize=12, color=COLORS['green'],
        fontweight='bold', rotation=-45)

ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Logistic Regression: Decision Boundary with Probability Gradient')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig5_13_logistic_regression_2d')
plt.show()

In [ ]:
print('All Chapter 5 figures generated.')